# Sequence Inspector

Randomly samples 20 `.npz` files from `processed/sequences/` and, for each,
plots the **reconstructed segment** (all windows stitched at stride spacing)
alongside a **zoomed view of one selected window**.

**What is the reconstructed segment?**  
The `.npz` stores only the saved Class-A windows (`shape: (N, 1024, 1)`,
MAD-normalised, NaN-free). The original unsaved segment is re-approximated
by placing each window at its stride offset. The overlap zone takes the value
from the later window — this is a visualisation approximation, not the exact
pipeline output, but it faithfully represents what the model sees.

In [ ]:
import glob
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
SEQ_DIR   = Path("../../processed/sequences")
T         = 1024    # window length (cadences)
STRIDE    = 512     # stride used when building windows
N_SAMPLE  = 20
SEED      = 42

# ── Sample ────────────────────────────────────────────────────────────────────
all_files = sorted(SEQ_DIR.glob("**/*.npz"))
print(f"Total .npz files found: {len(all_files)}")

rng = random.Random(SEED)
sample_files = rng.sample(all_files, min(N_SAMPLE, len(all_files)))
print(f"Sampled {len(sample_files)} files (seed={SEED})")

In [ ]:
def reconstruct_segment(windows: np.ndarray, stride: int) -> np.ndarray:
    """Stitch Class-A windows back into an approximate segment.

    windows: (N, T) float array, already squeezed.
    stride:  cadence stride used during windowing.
    Returns a 1-D float array of length (N-1)*stride + T.
    """
    n, t = windows.shape
    total = (n - 1) * stride + t
    seg = np.full(total, np.nan)
    for i, w in enumerate(windows):
        seg[i * stride : i * stride + t] = w
    return seg


def plot_seg_vs_window(npz_path: Path, window_idx: int = 0) -> None:
    d       = np.load(npz_path, allow_pickle=True)
    windows = d["windows"].squeeze(-1)   # (N, 1024, 1) -> (N, 1024)
    tic_id  = int(d["tic_id"])
    sector  = int(d["sector"])
    seg_idx = int(d["seg_idx"])
    n_win   = windows.shape[0]

    window_idx = min(window_idx, n_win - 1)

    seg         = reconstruct_segment(windows, STRIDE)   # 1-D
    cadences    = np.arange(len(seg))

    win_start   = window_idx * STRIDE
    win_end     = win_start + T
    win_flux    = windows[window_idx]                    # (1024,)
    win_cad     = cadences[win_start:win_end]

    print(
        f"TIC {tic_id}  sector={sector}  seg={seg_idx}  "
        f"n_windows={n_win}  seg_len={len(seg)}  "
        f"showing window {window_idx}  [{win_start}, {win_end})"
    )

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

    # Top: full reconstructed segment with selected window highlighted
    axes[0].plot(cadences, seg, lw=0.5, color="steelblue", alpha=0.85)
    axes[0].axvspan(
        win_start, win_end - 1, color="orange", alpha=0.30,
        label=f"Window {window_idx}  [{win_start}, {win_end})"
    )
    axes[0].set_title(
        f"TIC {tic_id}  |  Sector {sector}  |  Segment {seg_idx}  "
        f"({n_win} windows, {len(seg)} cadences)",
        fontsize=11
    )
    axes[0].set_ylabel("MAD-normalised flux")
    axes[0].legend(fontsize=9, loc="upper right")
    axes[0].set_xlabel("Cadence offset within segment")

    # Bottom: zoomed window
    axes[1].plot(win_cad, win_flux, lw=0.8, color="darkorange", marker=".", ms=1.5)
    axes[1].set_title(
        f"Window {window_idx}  |  cadences [{win_start}, {win_end})  |  "
        f"length={T}  NaN={int(np.isnan(win_flux).sum())}",
        fontsize=11
    )
    axes[1].set_ylabel("MAD-normalised flux")
    axes[1].set_xlabel("Cadence offset within segment")

    plt.tight_layout()
    plt.show()

In [ ]:
WINDOW_IDX = 0   # which window to highlight; clipped to [0, n_windows-1]

for path in sample_files:
    plot_seg_vs_window(path, window_idx=WINDOW_IDX)